In [2]:
import os
import sys 
import torch
from pathlib import Path

from satchange.utils import load_config
from satchange.data import download_levir_cd
from satchange.prepare_levir import prepare_levir

In [3]:
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [4]:


print("Python executable:", sys.executable)
print("Python version:", sys.version)

print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("Selected GPU:", torch.cuda.get_device_name(0))

Python executable: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/.venv/bin/python
Python version: 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:40:35) [GCC 12.3.0]
PyTorch: 2.12.1+cu130
PyTorch CUDA: 13.0
CUDA available: True
Visible GPU count: 1
Selected GPU: NVIDIA A100-SXM4-80GB


In [7]:
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

config_path = project_root / "configs" / "fc_siam_diff.yaml"

print("Current directory:", current_directory)
print("Project root:", project_root)
print("Config path:", config_path)
print("Config exists:", config_path.exists())

Current directory: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/notebooks
Project root: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence
Config path: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/configs/fc_siam_diff.yaml
Config exists: True


In [8]:
config = load_config(config_path)

config

{'project': {'name': 'satellite-change-intelligence', 'seed': 42},
 'runtime': {'device': 'auto', 'mixed_precision': True},
 'model': {'name': 'fc_siam_diff',
  'parameters': {'in_channels': 3,
   'out_channels': 1,
   'base_channels': [16, 32]}},
 'data': {'image_size': 256, 'batch_size': 2, 'num_workers': 0},
 'training': {'epochs': 100,
  'learning_rate': 0.001,
  'weight_decay': 0.0001,
  'loss': {'bce_weight': 0.5, 'dice_weight': 0.5},
  'threshold': 0.5},
 'output': {'directory': 'artifacts/fc_siam_diff',
  'checkpoint_name': 'best.pt'},
 'dataset': {'name': 'levir_cd',
  'download_url': 'https://drive.google.com/drive/folders/1dLuzldMRmbBNKPpUkX8Z53hi6NHLrWim',
  'download_directory': 'data/downloads/levir_cd',
  'prepared_directory': 'data/LEVIR-CD'}}

In [8]:
print("Model:", config["model"]["name"])
print("Image size:", config["data"]["image_size"])
print("Batch size:", config["data"]["batch_size"])
print("Epochs:", config["training"]["epochs"])

Model: fc_siam_diff
Image size: 256
Batch size: 2
Epochs: 100


In [10]:
dataset_config = config["dataset"]

dataset_name = dataset_config["name"]
dataset_url = dataset_config["download_url"]

download_directory = (
    project_root / dataset_config["download_directory"]
)

prepared_directory = (
    project_root / dataset_config["prepared_directory"]
)

print("Dataset name:", dataset_name)
print("Dataset URL:", dataset_url)
print("Download directory:", download_directory)
print("Prepared directory:", prepared_directory)

Dataset name: levir_cd
Dataset URL: https://drive.google.com/drive/folders/1dLuzldMRmbBNKPpUkX8Z53hi6NHLrWim
Download directory: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/downloads/levir_cd
Prepared directory: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/LEVIR-CD


In [12]:
downloaded_dataset_directory = download_levir_cd(
    download_url=dataset_url,
    download_directory=download_directory,
)

print(
    "Download dataset directory:",
    downloaded_dataset_directory,
)

Retrieving folder contents


Retrieving folder 1C2B5_wiZVUt-PWHn14H9lSKF5sA6jowR BIT_LEVIR
Processing file 1IVdF5a3e1_7DiSndtMkhpZuCSgDLLFcg best_ckpt.pt
Retrieving folder 1QZOYP-N2Nr27aDG38zTVJMunfzmrseLv PAM-pretrain
Processing file 196bR8LdSfVawnSOVmM_YPW25UetucUYX pam_net_A.pth
Processing file 1QOjoa-YFvHOgCLXaxvZUlrlidihpmbWU pam_net_F.pth
Processing file 1jj3qJD_grJlgIhUWO09zibRGJe0R4Tn0 test.zip
Processing file 18GuoCuBn48oZKAlEo-LrNwABrFhVALU- train.zip
Processing file 1BqSt4ueO7XAyQ_84mUjswUSJt13ZBuzG val.zip


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1IVdF5a3e1_7DiSndtMkhpZuCSgDLLFcg
To: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/downloads/levir_cd/BIT_LEVIR/best_ckpt.pt
100%|██████████| 60.0M/60.0M [00:01<00:00, 36.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=196bR8LdSfVawnSOVmM_YPW25UetucUYX
To: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/downloads/levir_cd/PAM-pretrain/pam_net_A.pth
100%|██████████| 162k/162k [00:00<00:00, 5.33MB/s]
Downloading...
From: https://drive.google.com/uc?id=1QOjoa-YFvHOgCLXaxvZUlrlidihpmbWU
To: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/downloads/levir_cd/PAM-pretrain/pam_net_F.pth
100%|██████████| 67.7M/67.7M [00:01<00:00, 62.4MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1jj3qJD_grJlgIhUWO09zibRGJe0R4Tn0
From (

Download dataset directory: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/downloads/levir_cd


Download completed


In [9]:
dataset_root = prepare_levir(
    config=config,
    project_root=project_root,
)

print(f"Ready dataset root: {dataset_root}")

Download directory is not empty: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/downloads/levir_cd
Skipping download.
Extracting: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/downloads/levir_cd/test.zip
Extracting: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/downloads/levir_cd/val.zip
Extracting: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/downloads/levir_cd/train.zip

Split:         complete dataset
Directory:     /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/LEVIR-CD
Before images: 637
After images:  637
Labels:        637

LEVIR-CD structure is valid.
Ready dataset root: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/LEVIR-CD
